# Data wrangling on Google Play Store data

The cells below clean a dataset of Android apps. The code comes from popular Kaggle notebooks, collected in
Yang et al., *Subtle Bugs Everywhere: Generating Documentation for Data Wrangling Code*, ASE 2021, and adapted to one dataset.

Every cell runs without an exception. Does every cell do what it is supposed to do?

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('data/googleplaystore.csv')
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Pixel Music Player 4,MEDICAL,4.8,35,19M,500+,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up
1,Bright Calendar,FAMILY,4.1,642,11M,"50,000+",Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device
2,Happy Chess,SOCIAL,4.2,151,10M,"10,000+",Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up
3,Mini Calendar,SHOPPING,3.6,160,Varies with device,"10,000+",Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up
4,Pocket Translator,LIFESTYLE,4.4,756,41M,"50,000+",Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up


In [2]:
df.shape

(2500, 13)

## Size: 'Varies with device' to missing value, '19M' and '201k' to numbers

In [3]:
# first change 'Varies with device' to nan
df['Size'] = df['Size'].replace('Varies with device', np.nan)

# convert Size
num = df.Size.replace(r'[kM]+$', '', regex=True).astype(float)
factor = df.Size.str.extract(r'[\d\.]+([KM]+)', expand=False)
factor = factor.replace(['k','M'], [10**3, 10**6]).fillna(1)
df['Size'] = num * factor.astype(int)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Pixel Music Player 4,MEDICAL,4.8,35,19000000.0,500+,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up
1,Bright Calendar,FAMILY,4.1,642,11000000.0,"50,000+",Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device
2,Happy Chess,SOCIAL,4.2,151,10000000.0,"10,000+",Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up
3,Mini Calendar,SHOPPING,3.6,160,NaN,"10,000+",Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up
4,Pocket Translator,LIFESTYLE,4.4,756,41000000.0,"50,000+",Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up


## Fill missing sizes with the mean size

In [4]:
df['Size'].fillna(df['Size'].mean(), inplace=True)
df.head()

/tmp/ipykernel_507/3124087065.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Size'].fillna(df['Size'].mean(), inplace=True)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Pixel Music Player 4,MEDICAL,4.8,35,19000000.0,500+,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up
1,Bright Calendar,FAMILY,4.1,642,11000000.0,"50,000+",Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device
2,Happy Chess,SOCIAL,4.2,151,10000000.0,"10,000+",Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up
3,Mini Calendar,SHOPPING,3.6,160,NaN,"10,000+",Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up
4,Pocket Translator,LIFESTYLE,4.4,756,41000000.0,"50,000+",Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up


## Year of the last update (without apps that have no update date)

In [5]:
df['Update_year'] = df['Last Updated'].dropna().map(lambda x: x.split(',')[1].split(' ')[1])
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Update_year
0,Pixel Music Player 4,MEDICAL,4.8,35,19000000.0,500+,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up,2015
1,Bright Calendar,FAMILY,4.1,642,11000000.0,"50,000+",Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device,2015
2,Happy Chess,SOCIAL,4.2,151,10000000.0,"10,000+",Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up,2017
3,Mini Calendar,SHOPPING,3.6,160,NaN,"10,000+",Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up,2017
4,Pocket Translator,LIFESTYLE,4.4,756,41000000.0,"50,000+",Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up,2016


## Fill missing ratings with the mean rating of the app's category

In [6]:
map_means = df.groupby('Category')['Rating'].mean()
idx_nan_rating = df.loc[df.Rating.isnull()].index
df.loc[idx_nan_rating, 'Rating'].loc[idx_nan_rating] = df['Category'].loc[idx_nan_rating].map(map_means)
df.head()

/tmp/ipykernel_507/2481114096.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df.loc[idx_nan_rating, 'Rating'].loc[idx_nan_rating] = df['Category'].loc[idx_nan_rating].map(map_means)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Update_year
0,Pixel Music Player 4,MEDICAL,4.8,35,19000000.0,500+,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up,2015
1,Bright Calendar,FAMILY,4.1,642,11000000.0,"50,000+",Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device,2015
2,Happy Chess,SOCIAL,4.2,151,10000000.0,"10,000+",Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up,2017
3,Mini Calendar,SHOPPING,3.6,160,NaN,"10,000+",Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up,2017
4,Pocket Translator,LIFESTYLE,4.4,756,41000000.0,"50,000+",Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up,2016


## Installs: '10,000+' to 10000

In [7]:
df['Installs'] = df['Installs'].str.replace(',', '').str.replace('+', '')
df['Installs'].astype(str).astype(int)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Update_year
0,Pixel Music Player 4,MEDICAL,4.8,35,19000000.0,500,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up,2015
1,Bright Calendar,FAMILY,4.1,642,11000000.0,50000,Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device,2015
2,Happy Chess,SOCIAL,4.2,151,10000000.0,10000,Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up,2017
3,Mini Calendar,SHOPPING,3.6,160,NaN,10000,Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up,2017
4,Pocket Translator,LIFESTYLE,4.4,756,41000000.0,50000,Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up,2016


## Reviews: '54k' and '2.1M' to numbers

In [8]:
df['Reviews'] = df['Reviews'].replace(regex=['k'], value='000')
df['Reviews'] = df['Reviews'].replace(regex=['M'], value='000000')
df['Reviews'] = df['Reviews'].astype(str).astype(float)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Update_year
0,Pixel Music Player 4,MEDICAL,4.8,35.0,19000000.0,500,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up,2015
1,Bright Calendar,FAMILY,4.1,642.0,11000000.0,50000,Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device,2015
2,Happy Chess,SOCIAL,4.2,151.0,10000000.0,10000,Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up,2017
3,Mini Calendar,SHOPPING,3.6,160.0,NaN,10000,Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up,2017
4,Pocket Translator,LIFESTYLE,4.4,756.0,41000000.0,50000,Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up,2016


## Reviews as integers

In [9]:
df['Reviws'] = df['Reviews'].apply(int)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Update_year,Reviws
0,Pixel Music Player 4,MEDICAL,4.8,35.0,19000000.0,500,Free,0,Everyone,Medical,"December 11, 2015",1.17.1,4.1 and up,2015,35
1,Bright Calendar,FAMILY,4.1,642.0,11000000.0,50000,Free,0,Everyone,Casual,"February 4, 2015",3.19.8,Varies with device,2015,642
2,Happy Chess,SOCIAL,4.2,151.0,10000000.0,10000,Free,0,Everyone,Social,"November 14, 2017",5.8.0,5.0 and up,2017,151
3,Mini Calendar,SHOPPING,3.6,160.0,NaN,10000,Free,0,Everyone,Shopping,"October 17, 2017",5.16.5,4.1 and up,2017,160
4,Pocket Translator,LIFESTYLE,4.4,756.0,41000000.0,50000,Free,0,Everyone,Lifestyle,"August 30, 2016",3.18.5,4.4 and up,2016,756


## Cleaned data

In [10]:
df.describe()

,Rating,Reviews,Size,Reviws
count,2293.000000,2500.000000,2.143000e+03,2500.000000
mean,4.196468,19160.981920,1.894501e+07,19160.941600
std,0.435089,74181.821725,1.904817e+07,74181.832126
min,2.000000,0.000000,2.300000e+01,0.000000
25%,3.900000,58.875000,6.600000e+06,58.750000
50%,4.200000,540.000000,1.300000e+07,540.000000
75%,4.500000,4539.500000,2.400000e+07,4539.500000
max,5.000000,951738.000000,1.000000e+08,951738.000000
